In [2]:
# CELL 1: DB SETUP
from flask import Flask
from flask_sqlalchemy import SQLAlchemy
from datetime import datetime
import os

# Initialize Flask app and configure SQLite
app = Flask(__name__)
db_path = os.path.join(os.getcwd(), 'study_companion.db')
app.config['SQLALCHEMY_DATABASE_URI'] = f'sqlite:///{db_path}'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False

db = SQLAlchemy(app)

# Define the Tables
class MasterCatalog(db.Model):
    __tablename__ = 'master_catalog'
    catalog_id = db.Column(db.Integer, primary_key=True)
    subject_name = db.Column(db.String(100), nullable=False)
    topic_name = db.Column(db.String(200), nullable=False)
    category = db.Column(db.String(50), nullable=False)

class MySyllabus(db.Model):
    __tablename__ = 'my_syllabus'
    id = db.Column(db.Integer, primary_key=True)
    catalog_id = db.Column(db.Integer, db.ForeignKey('master_catalog.catalog_id'), nullable=False)
    status = db.Column(db.String(50), default='Pending')
    priority = db.Column(db.String(50), default='Medium')

class DailyLog(db.Model):
    __tablename__ = 'daily_logs'
    id = db.Column(db.Integer, primary_key=True)
    date = db.Column(db.Date, default=datetime.utcnow().date, unique=True, nullable=False)
    sleep_hours = db.Column(db.Float, nullable=False)
    sleep_quality_flag = db.Column(db.Boolean, nullable=False)
    difficulty_rating_prev = db.Column(db.Integer, nullable=False)
    quiz_score = db.Column(db.Integer, nullable=False)
    days_to_exam = db.Column(db.Integer, nullable=False)
    energy_level = db.Column(db.Integer, nullable=False)
    xgboost_mode_label = db.Column(db.String(50), nullable=False)
    actual_plan_heuristic = db.Column(db.Text, nullable=False)

# Create the .db file
with app.app_context():
    db.create_all()
    print("✅ Database and tables created successfully!")

✅ Database and tables created successfully!


C:\Users\divya\AppData\Local\Temp\ipykernel_14020\2192082900.py:33: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  date = db.Column(db.Date, default=datetime.utcnow().date, unique=True, nullable=False)


In [3]:
# CELL 2: SEED DATA & ONBOARDING
def seed_and_pick():
    subjects = [
        ('DBMS', 'Normalization', 'Theory'),
        ('DBMS', 'SQL Joins', 'Coding'),
        ('DSA', 'Binary Search Trees', 'Coding'),
        ('OS', 'Process Scheduling', 'Theory')
    ]
    
    with app.app_context():
        # 1. Seed Master Catalog
        if MasterCatalog.query.count() == 0:
            for sub, top, cat in subjects:
                db.session.add(MasterCatalog(subject_name=sub, topic_name=top, category=cat))
            db.session.commit()
            print("✅ Master Catalog seeded.")
            
        # 2. Pick 'DBMS' and 'DSA' for your syllabus
        for subject in ['DBMS', 'DSA']:
            topics = MasterCatalog.query.filter_by(subject_name=subject).all()
            for t in topics:
                exists = MySyllabus.query.filter_by(catalog_id=t.catalog_id).first()
                if not exists:
                    db.session.add(MySyllabus(catalog_id=t.catalog_id, status='Pending'))
        db.session.commit()
        print("✅ Added DBMS and DSA to MySyllabus.")

seed_and_pick()

✅ Master Catalog seeded.
✅ Added DBMS and DSA to MySyllabus.


In [4]:
# CELL 3: DUMMY DATA GENERATION
import pandas as pd
import numpy as np

def generate_dummy_history(rows=1000):
    np.random.seed(42) # Keeps the random data consistent
    data = []
    for _ in range(rows):
        energy = np.random.randint(1, 6)
        sleep = np.random.uniform(4, 9)
        exam_dist = np.random.randint(1, 30)
        quiz = np.random.randint(40, 100)
        
        # Rule-based labeling to teach the AI
        if energy <= 2 or sleep < 5:
            label = 0 # Recovery
        elif quiz < 60:
            label = 1 # Revision
        elif energy >= 4 and exam_dist > 7:
            label = 3 # Deep Work
        else:
            label = 2 # Standard
            
        data.append([sleep, energy, exam_dist, quiz, label])
    
    return pd.DataFrame(data, columns=['sleep_hours', 'energy_level', 'days_to_exam', 'quiz_score', 'label'])

df_history = generate_dummy_history()
print("✅ Generated 1000 rows of dummy study history.")
df_history.head()

✅ Generated 1000 rows of dummy study history.


,sleep_hours,energy_level,days_to_exam,quiz_score,label
0,8.753572,4,11,47,1
1,6.984251,5,26,58,1
2,6.296244,3,21,75,2
3,4.102922,3,2,63,0
4,8.692764,4,2,99,2


In [5]:
# CELL 4: XGBOOST TRAINING
import xgboost as xgb
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Split Data
X = df_history[['sleep_hours', 'energy_level', 'days_to_exam', 'quiz_score']]
y = df_history['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Initialize and Train Model
model = xgb.XGBClassifier(max_depth=3, learning_rate=0.1, n_estimators=50, objective='multi:softmax', num_class=4)
model.fit(X_train, y_train)

# 3. Test Accuracy
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"✅ Model Accuracy: {accuracy * 100:.2f}%")

# 4. Save the Model
model_path = os.path.join(os.getcwd(), 'xgboost_v1.pkl')
joblib.dump(model, model_path)
print(f"💾 Model saved successfully to: {model_path}")

✅ Model Accuracy: 100.00%
💾 Model saved successfully to: c:\projects\adaptive ai companion\notebooks\xgboost_v1.pkl
